<a href="https://colab.research.google.com/github/AliefReefandi/Grokking-Deep-Learning/blob/main/Chapter_12_Neural_Networks_that_Write_Like_Shakespeare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 12 — Neural Networks That Write Like Shakespeare

## Recurrent Layers for Variable-Length Data

Chapter ini membahas bagaimana neural network dapat memproses data dengan panjang yang berbeda-beda menggunakan **Recurrent Neural Network (RNN)**.

Fokus utama chapter ini adalah membuat representasi vektor untuk sequence dengan panjang arbitrer dan menggunakan representasi tersebut untuk memprediksi kata berikutnya.

## Tujuan Pembelajaran

Setelah menyelesaikan chapter ini, kita diharapkan dapat:

1. Memahami permasalahan data dengan panjang yang berbeda-beda.
2. Memahami konsep sentence embedding.
3. Memahami keterbatasan pendekatan bag-of-words.
4. Memahami penggunaan transition matrix pada RNN.
5. Memahami forward propagation pada sequence.
6. Memahami backpropagation pada sequence dengan panjang arbitrer.
7. Memahami weight update pada RNN.
8. Memahami penggunaan RNN untuk memprediksi kata berikutnya.
9. Memahami konsep perplexity sebagai ukuran performa language model.

## 1. The Challenge of Arbitrary Length

Pada chapter sebelumnya, word embedding digunakan untuk merepresentasikan sebuah kata dalam bentuk vektor.

Namun, sebuah kalimat terdiri dari banyak kata dan jumlah kata pada setiap kalimat dapat berbeda.

Contoh:

- "the cat sat"
- "the cat sat still"

Jika embedding setiap kata langsung digabungkan (concatenation), maka panjang vektor yang dihasilkan juga berbeda.

Hal ini membuat representasi kalimat sulit dibandingkan satu sama lain.

Karena itu, dibutuhkan cara untuk mengubah sequence dengan panjang berbeda menjadi representasi dengan ukuran tetap.

## 2. Averaged Word Vectors

Salah satu pendekatan sederhana adalah mengambil rata-rata embedding dari seluruh kata dalam sebuah kalimat.

Dengan cara ini, berapa pun jumlah kata dalam kalimat, hasil akhirnya tetap berupa satu vektor dengan ukuran yang sama.

Contoh:

"The cat sat"

akan direpresentasikan sebagai:

average(
    embedding("the"),
    embedding("cat"),
    embedding("sat")
)

Pendekatan ini cukup berguna karena kata-kata dengan makna yang mirip dapat menghasilkan representasi yang mirip.

Namun, pendekatan ini memiliki kelemahan penting: urutan kata tidak dipertahankan.

## 3. Limitations of Bag-of-Words

Bag-of-words merepresentasikan teks berdasarkan kata-kata yang muncul tanpa memperhatikan urutannya.

Contoh:

"Red Sox defeat Yankees"

dan

"Yankees defeat Red Sox"

dapat menghasilkan representasi yang sama jika hanya memperhatikan kumpulan katanya.

Padahal kedua kalimat tersebut memiliki makna yang berbeda.

Karena itu, model perlu mempertahankan informasi mengenai urutan kata.

## 4. Transition Matrix

Untuk mempertahankan informasi urutan kata, chapter ini memperkenalkan transition matrix.

Awalnya digunakan identity matrix.

Identity matrix tidak mengubah nilai vector ketika dilakukan perkalian:

vector × identity = vector

Kemudian identity matrix tersebut dapat dipelajari oleh neural network.

Matrix ini digunakan kembali pada setiap perpindahan dari satu kata ke kata berikutnya.

Dengan demikian, model dapat mempelajari bagaimana informasi dari kata sebelumnya memengaruhi representasi kata berikutnya.

In [1]:
import numpy as np

def softmax(x_):
    x = np.atleast_2d(x_)
    temp = np.exp(x)
    return temp / np.sum(temp, axis=1, keepdims=True)

word_vects = {}

word_vects['yankees'] = np.array([[0., 0., 0.]])
word_vects['bears'] = np.array([[0., 0., 0.]])
word_vects['braves'] = np.array([[0., 0., 0.]])
word_vects['red'] = np.array([[0., 0., 0.]])
word_vects['sox'] = np.array([[0., 0., 0.]])
word_vects['lose'] = np.array([[0., 0., 0.]])
word_vects['defeat'] = np.array([[0., 0., 0.]])
word_vects['beat'] = np.array([[0., 0., 0.]])
word_vects['tie'] = np.array([[0., 0., 0.]])

sent2output = np.random.rand(
    3,
    len(word_vects)
)

identity = np.eye(3)

## 5. Forward Propagation

Contoh sequence yang digunakan:

"red sox defeat"

Target:

"yankees"

Prosesnya:

red → sox → defeat → prediction

Setiap kata menghasilkan hidden representation baru melalui transition matrix.

In [2]:
layer_0 = word_vects['red']

layer_1 = (
    layer_0.dot(identity)
    + word_vects['sox']
)

layer_2 = (
    layer_1.dot(identity)
    + word_vects['defeat']
)

pred = softmax(
    layer_2.dot(sent2output)
)

print(pred)

[[0.11111111 0.11111111 0.11111111 0.11111111 0.11111111 0.11111111
  0.11111111 0.11111111 0.11111111]]


## 6. Backpropagation pada Sequence

Backpropagation pada RNN tetap menggunakan prinsip yang sama seperti neural network sebelumnya.

Perbedaannya adalah error harus mengalir kembali melewati beberapa timestep.

Jika terdapat sequence:

red → sox → defeat → Yankees

maka error dari prediction akan digunakan untuk memperbarui:

- word embedding "defeat"
- word embedding "sox"
- word embedding "red"
- transition matrix
- output weights

In [3]:
y = np.array([
    1, 0, 0, 0, 0,
    0, 0, 0, 0
])

pred_delta = pred - y

layer_2_delta = (
    pred_delta.dot(sent2output.T)
)

defeat_delta = layer_2_delta * 1

layer_1_delta = (
    layer_2_delta.dot(identity.T)
)

sox_delta = layer_1_delta * 1

layer_0_delta = (
    layer_1_delta.dot(identity.T)
)

In [5]:
alpha = 0.01

word_vects['red'] -= (
    layer_0_delta * alpha
)

word_vects['sox'] -= (
    sox_delta * alpha
)

word_vects['defeat'] -= (
    defeat_delta * alpha
)

identity -= (
    np.outer(
        layer_0,
        layer_1_delta
    ) * alpha
)

identity -= (
    np.outer(
        layer_1,
        layer_2_delta
    ) * alpha
)

sent2output -= (
    np.outer(
        layer_2,
        pred_delta
    ) * alpha
)

## 7. Training dengan Dataset Babi

Setelah memahami konsep dasar, chapter ini menggunakan dataset Babi sebagai toy dataset.

Dataset ini berisi kalimat sederhana seperti:

- Mary moved to the bathroom
- John went to the hallway
- Where is Mary bathroom

Untuk eksperimen ini, model digunakan untuk mencoba memprediksi kata berikutnya berdasarkan kata-kata sebelumnya.

In [9]:
!wget -q https://s3.amazonaws.com/text-datasets/babi_tasks_1-20_v1-2.tar.gz
!tar -xvf babi_tasks_1-20_v1-2.tar.gz

tasks_1-20_v1-2/
tasks_1-20_v1-2/en-10k/
tasks_1-20_v1-2/en-10k/qa12_conjunction_train.txt
tasks_1-20_v1-2/en-10k/qa11_basic-coreference_test.txt
tasks_1-20_v1-2/en-10k/qa4_two-arg-relations_train.txt
tasks_1-20_v1-2/en-10k/qa20_agents-motivations_train.txt
tasks_1-20_v1-2/en-10k/qa18_size-reasoning_test.txt
tasks_1-20_v1-2/en-10k/qa5_three-arg-relations_test.txt
tasks_1-20_v1-2/en-10k/qa12_conjunction_test.txt
tasks_1-20_v1-2/en-10k/qa2_two-supporting-facts_test.txt
tasks_1-20_v1-2/en-10k/qa17_positional-reasoning_test.txt
tasks_1-20_v1-2/en-10k/qa14_time-reasoning_test.txt
tasks_1-20_v1-2/en-10k/qa6_yes-no-questions_test.txt
tasks_1-20_v1-2/en-10k/qa17_positional-reasoning_train.txt
tasks_1-20_v1-2/en-10k/qa15_basic-deduction_test.txt
tasks_1-20_v1-2/en-10k/qa13_compound-coreference_train.txt
tasks_1-20_v1-2/en-10k/qa3_three-supporting-facts_test.txt
tasks_1-20_v1-2/en-10k/qa13_compound-coreference_test.txt
tasks_1-20_v1-2/en-10k/qa15_basic-deduction_train.txt
tasks_1-20_v1-2/en-10k/

In [10]:
import os

print(os.path.exists("babi_tasks_1-20_v1-2.tar.gz"))
print(os.path.exists("tasks_1-20_v1-2"))

True
True


In [12]:
!git clone -q --depth 1 https://github.com/iamtrask/Grokking-Deep-Learning.git

In [13]:
import sys
import random
import math
from collections import Counter
import numpy as np

f = open(
    'Grokking-Deep-Learning/tasksv11/en/qa1_single-supporting-fact_train.txt',
    'r'
)

raw = f.readlines()
f.close()

tokens = list()

for line in raw[0:1000]:
    tokens.append(
        line.lower()
        .replace("\n", "")
        .split(" ")[1:]
    )

print(tokens[0:3])

[['mary', 'moved', 'to', 'the', 'bathroom.'], ['john', 'went', 'to', 'the', 'hallway.'], ['where', 'is', 'mary?', '\tbathroom\t1']]


## 8. Setting Up the RNN

Sebelum melakukan training, kita perlu membuat:

- vocabulary
- word-to-index mapping
- fungsi untuk mengubah kata menjadi index
- fungsi softmax
- word embeddings
- recurrent matrix
- start vector
- decoder matrix
- one-hot matrix

In [14]:
vocab = set()

for sent in tokens:
    for word in sent:
        vocab.add(word)

vocab = list(vocab)

word2index = {}

for i, word in enumerate(vocab):
    word2index[word] = i


def words2indices(sentence):
    idx = list()

    for word in sentence:
        idx.append(word2index[word])

    return idx


def softmax(x):
    e_x = np.exp(
        x - np.max(x)
    )

    return e_x / e_x.sum(
        axis=0
    )

In [15]:
np.random.seed(1)

embed_size = 10

embed = (
    np.random.rand(
        len(vocab),
        embed_size
    ) - 0.5
) * 0.1

recurrent = np.eye(embed_size)

start = np.zeros(embed_size)

decoder = (
    np.random.rand(
        embed_size,
        len(vocab)
    ) - 0.5
) * 0.1

one_hot = np.eye(len(vocab))

print("Vocabulary size:", len(vocab))
print("Embedding shape:", embed.shape)
print("Recurrent shape:", recurrent.shape)
print("Decoder shape:", decoder.shape)

Vocabulary size: 82
Embedding shape: (82, 10)
Recurrent shape: (10, 10)
Decoder shape: (10, 82)


## 9. Forward Propagation dengan Arbitrary Length

Berbeda dari neural network sebelumnya, jumlah timestep bergantung pada panjang sequence.

Oleh karena itu, digunakan sebuah list bernama `layers`.

Setiap timestep akan memiliki layer sendiri.

Pada setiap timestep:

1. Model melakukan prediction.
2. Loss dihitung.
3. Hidden state berikutnya dibuat.
4. Hidden state digunakan untuk timestep berikutnya.

Dengan pendekatan ini, model dapat memproses sequence dengan panjang yang berbeda.

In [16]:
def predict(sent):

    layers = list()

    layer = {}

    layer['hidden'] = start

    layers.append(layer)

    loss = 0

    preds = list()

    for target_i in range(len(sent)):

        layer = {}

        layer['pred'] = softmax(
            layers[-1]['hidden'].dot(
                decoder
            )
        )

        loss += -np.log(
            layer['pred'][sent[target_i]]
        )

        layer['hidden'] = (
            layers[-1]['hidden'].dot(
                recurrent
            )
            + embed[sent[target_i]]
        )

        layers.append(layer)

    return layers, loss

In [17]:
test_sentence = words2indices(
    tokens[0][1:]
)

layers, loss = predict(
    test_sentence
)

print("Jumlah timestep:", len(layers))
print("Loss:", loss)

Jumlah timestep: 5
Loss: 17.623742576359106


## 10. Backpropagation dengan Arbitrary Length

Pada RNN, gradient harus bergerak mundur melewati setiap timestep.

Proses ini dilakukan dengan:

`reversed(range(len(layers)))`

Dengan demikian, model melakukan backpropagation dari timestep terakhir menuju timestep pertama.

Gradient pada hidden state juga dapat menerima kontribusi dari timestep berikutnya.

In [18]:
for iter in range(30000):

    alpha = 0.001

    sent = words2indices(
        tokens[iter % len(tokens)][1:]
    )

    layers, loss = predict(sent)

    for layer_idx in reversed(
        range(len(layers))
    ):

        layer = layers[layer_idx]

        target = sent[layer_idx - 1]

        if layer_idx > 0:

            layer['output_delta'] = (
                layer['pred']
                - one_hot[target]
            )

            new_hidden_delta = (
                layer['output_delta']
                .dot(decoder.transpose())
            )

            if layer_idx == len(layers) - 1:

                layer['hidden_delta'] = (
                    new_hidden_delta
                )

            else:

                layer['hidden_delta'] = (
                    new_hidden_delta
                    +
                    layers[
                        layer_idx + 1
                    ]['hidden_delta']
                    .dot(
                        recurrent.transpose()
                    )
                )

        else:

            layer['hidden_delta'] = (
                layers[
                    layer_idx + 1
                ]['hidden_delta']
                .dot(
                    recurrent.transpose()
                )
            )

## 11. Weight Update

Setelah gradient diperoleh, parameter network diperbarui.

Parameter yang diperbarui adalah:

- `start`
- `decoder`
- `embed`
- `recurrent`

Learning rate yang digunakan adalah:

`alpha = 0.001`

In [19]:
start -= (
    layers[0]['hidden_delta']
    * alpha
    / float(len(sent))
)

for layer_idx, layer in enumerate(
    layers[1:]
):

    decoder -= (
        np.outer(
            layers[layer_idx]['hidden'],
            layer['output_delta']
        )
        * alpha
        / float(len(sent))
    )

    embed_idx = sent[layer_idx]

    embed[embed_idx] -= (
        layers[layer_idx]['hidden_delta']
        * alpha
        / float(len(sent))
    )

    recurrent -= (
        np.outer(
            layers[layer_idx]['hidden'],
            layer['hidden_delta']
        )
        * alpha
        / float(len(sent))
    )

In [20]:
for iter in range(1000):

    alpha = 0.001

    sent = words2indices(
        tokens[iter % len(tokens)][1:]
    )

    layers, loss = predict(sent)

    for layer_idx in reversed(
        range(len(layers))
    ):

        layer = layers[layer_idx]

        target = sent[layer_idx - 1]

        if layer_idx > 0:

            layer['output_delta'] = (
                layer['pred']
                - one_hot[target]
            )

            new_hidden_delta = (
                layer['output_delta']
                .dot(decoder.transpose())
            )

            if layer_idx == len(layers) - 1:

                layer['hidden_delta'] = (
                    new_hidden_delta
                )

            else:

                layer['hidden_delta'] = (
                    new_hidden_delta
                    +
                    layers[
                        layer_idx + 1
                    ]['hidden_delta']
                    .dot(
                        recurrent.transpose()
                    )
                )

        else:

            layer['hidden_delta'] = (
                layers[
                    layer_idx + 1
                ]['hidden_delta']
                .dot(
                    recurrent.transpose()
                )
            )

    start -= (
        layers[0]['hidden_delta']
        * alpha
        / float(len(sent))
    )

    for layer_idx, layer in enumerate(
        layers[1:]
    ):

        decoder -= (
            np.outer(
                layers[layer_idx]['hidden'],
                layer['output_delta']
            )
            * alpha
            / float(len(sent))
        )

        embed_idx = sent[layer_idx]

        embed[embed_idx] -= (
            layers[layer_idx]['hidden_delta']
            * alpha
            / float(len(sent))
        )

        recurrent -= (
            np.outer(
                layers[layer_idx]['hidden'],
                layer['hidden_delta']
            )
            * alpha
            / float(len(sent))
        )

    if iter % 100 == 0:
        print(
            "Iteration:",
            iter,
            "| Perplexity:",
            np.exp(loss / len(sent))
        )

Iteration: 0 | Perplexity: 81.93550166174496
Iteration: 100 | Perplexity: 81.93936719169288
Iteration: 200 | Perplexity: 81.98813924153009
Iteration: 300 | Perplexity: 81.93392590308693
Iteration: 400 | Perplexity: 81.95517761576782
Iteration: 500 | Perplexity: 82.05647476346365
Iteration: 600 | Perplexity: 81.91053772600806
Iteration: 700 | Perplexity: 82.04109597214187
Iteration: 800 | Perplexity: 82.22474384041313
Iteration: 900 | Perplexity: 82.13029608986857


## 12. Perplexity

Perplexity digunakan untuk melihat seberapa baik model memprediksi sequence.

Secara sederhana:

- Perplexity tinggi → prediksi model masih buruk.
- Perplexity rendah → prediksi model semakin sesuai dengan data.

Dalam eksperimen buku, perplexity mengalami penurunan selama training.

Contoh hasil:

82.09
81.87
81.53
...
4.13
4.07
4.01

Semakin mendekati 1 berarti distribusi probabilitas prediksi semakin dekat dengan target.

## 13. Melihat Prediksi Model

Selain melihat perplexity, kita dapat melihat secara langsung kata yang diprediksi oleh model.

Dengan demikian kita dapat mengetahui:

- kata mana yang berhasil diprediksi,
- kata mana yang salah,
- pola apa yang sudah dipelajari model.

In [21]:
sent_index = 4

l, _ = predict(
    words2indices(
        tokens[sent_index]
    )
)

print(tokens[sent_index])

for i, each_layer in enumerate(
    l[1:-1]
):

    input_word = tokens[sent_index][i]

    true = tokens[sent_index][i + 1]

    pred = vocab[
        each_layer['pred'].argmax()
    ]

    print(
        "Prev Input:" +
        input_word +
        " " * max(1, 12 - len(input_word)) +
        "True:" +
        true +
        " " * max(1, 15 - len(true)) +
        "Pred:" +
        pred
    )

['sandra', 'moved', 'to', 'the', 'garden.']
Prev Input:sandra      True:moved          Pred:the
Prev Input:moved       True:to             Pred:	garden	5
Prev Input:to          True:the            Pred:	garden	11
Prev Input:the         True:garden.        Pred:	garden	11


## Analisis Hasil

Pada awal training, prediksi model masih banyak yang salah karena parameter network masih mendekati kondisi awal.

Setelah training berlangsung, model mulai mempelajari pola hubungan antar-kata.

Namun, model tidak selalu dapat memprediksi kata dengan benar. Hal ini terjadi karena beberapa sequence membutuhkan konteks yang lebih panjang untuk menentukan kata berikutnya.

Chapter ini menunjukkan bahwa RNN dapat membuat representasi fixed-size dari sequence dengan panjang yang berbeda, tetapi representasi tersebut tidak menyimpan seluruh informasi sequence. Model harus belajar informasi mana yang perlu dipertahankan dan mana yang dapat dilupakan.

## Rangkuman

Chapter 12 memperkenalkan Recurrent Neural Network (RNN) untuk menangani sequence dengan panjang yang berbeda.

Konsep utama yang dipelajari:

- Arbitrary-length sequences
- Sentence embeddings
- Averaged word vectors
- Bag-of-words limitations
- Transition matrix
- Forward propagation
- Backpropagation through sequence
- Weight update
- Next-word prediction
- Perplexity

RNN menggunakan hidden state untuk membawa informasi dari timestep sebelumnya ke timestep berikutnya. Dengan menggunakan transition matrix yang dipelajari, network dapat membuat representasi sequence dengan ukuran tetap meskipun jumlah kata dalam sequence berbeda.

## Kesimpulan

Chapter 12 menunjukkan bagaimana konsep neural network dapat dikembangkan untuk memproses data sequence dengan panjang yang berbeda.

Berbeda dengan pendekatan bag-of-words yang mengabaikan urutan kata, RNN mempertahankan informasi sequence melalui recurrent atau transition matrix.

Melalui forward propagation, backpropagation, dan weight update, model dapat belajar memprediksi kata berikutnya berdasarkan kata-kata sebelumnya.

Konsep RNN pada chapter ini menjadi dasar untuk pembahasan yang lebih kompleks pada chapter berikutnya, terutama ketika framework deep learning dibuat secara otomatis dan ketika model menggunakan arsitektur yang lebih kuat seperti LSTM.